# 📈 Linear Regression Playground

Welcome to the **Linear Regression Forecasting Playground**. 

This notebook serves as an interactive environment to train, evaluate, and visualize our cluster-based Linear Regression model. Unlike the automated production scripts, this playground breaks down the pipeline into step-by-step modular blocks, allowing for deep-dive analysis of each stage.

### Key Architecture
- **Granularity**: 15-minute intervals.
- **Strategy**: One Linear Regression model per cluster (Shape/Profile based).
- **Target**: Consumption in kW.
- **Evaluation**: Business-oriented metrics (MAPE, WMAPE) aggregated at the portfolio level.

## 🛠️ Step 0: environment Setup

First, we ensure our environment can resolve project-level modules from the `src` directory.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Ensure project root is in path to import modular tools
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import specialized functions from our core library
from src.models.linear_regression import (
    load_processed_data, 
    preprocess_and_split, 
    train_models, 
    predict_models, 
    evaluate_models,
    save_cluster_artifacts
)
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

print("✅ Setup complete. project modules loaded.")

## 📂 Step 1: Data Loading

We load the parquet file generated by the preprocessing pipeline. This file contains all features, including lags, rolling windows, weather signals, and cluster assignments.

In [ ]:
dataset_path = os.path.join(PROJECT_ROOT, 'Datasets', 'processed_electricity_data.parquet')
df_long = load_processed_data(dataset_path)

print(f"\nLoaded {len(df_long):,} observations.")
df_long[['ClientID', 'Date', 'Consumption', 'Cluster', 'Consumer_Category']].head()

## ⚙️ Step 2: Feature Engineering & Data Splitting

This step performs several critical actions:
1. **Train/Test Split**: Uses data before 2014 for training and 2014 onwards for testing.
2. **Standardization**: 
   - Weather signals (HDH, CDH) are scaled globally.
   - Consumption is scaled **per client** using individual StandardScalers to handle different orders of magnitude between consumers.

In [ ]:
results = preprocess_and_split(df_long)

# Unpacking results for clarity
train_df, test_df, X_train, y_train, X_test, client_scalers, scaler_weather, feature_cols = results

print(f"\nFeature Engineering Complete.")
print(f"Training set rows: {len(X_train):,}")
print(f"Testing set rows:  {len(X_test):,}")

## 🧠 Step 3: Cluster-based Model Training

Instead of training one global model, we train one independent Linear Regression for each identified cluster (Shape Profiles). This allows the model to learn specific behaviors for residential vs. industrial consumers.

In [ ]:
cluster_models = train_models(X_train, y_train, train_df)

print(f"\n✅ Successfully trained {len(cluster_models)} cluster models.")

## 🔮 Step 4: Vectorized Day-Ahead Prediction

We apply the trained models to the test set. The prediction is vectorized per cluster for high performance. We also apply physical constraints (capping consumption at 0 kW).

In [ ]:
test_df = predict_models(cluster_models, test_df, X_test, client_scalers)

print("\nPredictions generated for the entire test horizon.")
test_df[['ClientID', 'Date', 'Predicted_Consumption_Scaled']].head()

## 📊 Step 5: Performance Evaluation

We inverse-transform the predictions back to the original kW scale and calculate error metrics. 
- **MAPE** (Mean Absolute Percentage Error): Sensitivity to percentage deviations.
- **WMAPE** (Weighted MAPE): Weighting errors by volume, crucial for business reporting.

In [ ]:
cluster_eval, summary = evaluate_models(test_df, client_scalers)
print("\nEvaluation Summary Table:")
display(summary)

## 🎨 Step 6: Visual Analytics

We visualize the aggregated portfolio load (Actual vs. Predicted) and analyze the error distribution across different time windows.

In [ ]:
# Plot the last 2 weeks (1344 steps) of the portfolio
plot_cluster_portfolio(cluster_eval, summary, model_label="Linear Regression")

# Perform time-period split analysis
analyze_time_periods(test_df)

## 💾 Step 7: Persisting Artifacts

Finally, we save the models and scalers so they can be used by the **Agent Chatbot** for live forecasting.

In [ ]:
client_clusters = df_long.drop_duplicates(subset=['ClientID']).set_index('ClientID')['Cluster'].to_dict()
save_path = os.path.join(PROJECT_ROOT, 'agent', 'artifacts')

save_cluster_artifacts(
    cluster_models, 
    client_scalers, 
    scaler_weather, 
    feature_cols, 
    client_clusters, 
    artifacts_dir=save_path
)

print("\n✅ All production artifacts saved to agent/artifacts/")